# CityPilot IA · Cuaderno de validación

**Objetivo:** usar el contenedor Jupyter como **cliente independiente** de la web y PostgreSQL. Completa las celdas, ejecútalas y guarda los resultados en el repositorio. No es un notebook decorativo: los 4 controles son obligatorios.

> Dentro del contenedor, la URL de Flask es `http://web:5000`; `localhost:5000` apuntaría al propio Jupyter. Los datos de los fixtures son de demostración; no confundirlos con respuestas vivas.

In [2]:
import os
import requests
import pandas as pd
from sqlalchemy import create_engine, text

WEB = os.environ.get("WEB_BASE_URL", "http://web:5000")
DB = os.environ["DATABASE_URL"]
print("API:", WEB)
print("Base configurada:", bool(DB))

API: http://web:5000
Base configurada: True


## 0. Salud de la arquitectura (pista resuelta)
Comprueba que Jupyter puede llegar a Flask y que Flask responde sobre la BD. Explica por escrito qué significa cada conexión.

In [3]:
respuesta = requests.get(f"{WEB}/api/health", timeout=5)
print(respuesta.status_code, respuesta.json())
assert respuesta.status_code == 200

200 {'db': 'ok', 'estado': 'ok', 'servicio': 'citypilot'}


## 1. API de datos públicos (obligatorio)
Con `requests`, recupera **Valencia → coordenadas → clima actual**. No uses lat/lon de memoria como respuesta final: debes sacarlas del JSON de Geocoding. Muestra ciudad, lat, lon, temperatura y precipitación. Comprueba qué haces si `results` está vacío.

En `docs/APIS_Y_CONTRATOS.md` tienes URLs y parámetros. Puedes poner los campos `name`, `count`, `countryCode`, `current`, `timezone` en diccionarios de parámetros.

In [4]:
import requests

geo_url = "https://geocoding-api.open-meteo.com/v1/search"
meteo_url = "https://api.open-meteo.com/v1/forecast"

# 1. Geocodificación: obtener las coordenadas reales de Valencia
params_geo = {
    "name": "Valencia",
    "count": 1,
    "countryCode": "ES",
    "language": "es",
    "format": "json"
}

respuesta_geo = requests.get(geo_url, params=params_geo, timeout=15)
respuesta_geo.raise_for_status()
datos_geo = respuesta_geo.json()

# Comprobar si la búsqueda ha devuelto resultados
if not datos_geo.get("results"):
    raise ValueError("No se han encontrado resultados para Valencia.")

# Extraer los datos del JSON
ciudad = datos_geo["results"][0]
nombre_ciudad = ciudad["name"]
latitud = ciudad["latitude"]
longitud = ciudad["longitude"]

# 2. Consultar la meteorología con las coordenadas obtenidas
params_meteo = {
    "latitude": latitud,
    "longitude": longitud,
    "current": "temperature_2m,precipitation",
    "timezone": "auto"
}

respuesta_meteo = requests.get(meteo_url, params=params_meteo, timeout=15)
respuesta_meteo.raise_for_status()
datos_meteo = respuesta_meteo.json()

# Extraer la meteorología actual
actual = datos_meteo["current"]
temperatura = actual["temperature_2m"]
precipitacion = actual["precipitation"]

# 3. Mostrar resultados
print("CIUDAD:", nombre_ciudad)
print("LATITUD:", latitud)
print("LONGITUD:", longitud)
print("TEMPERATURA:", temperatura, "°C")
print("PRECIPITACIÓN:", precipitacion, "mm")
print("ZONA HORARIA:", datos_meteo["timezone"])


CIUDAD: Valencia
LATITUD: 39.47391
LONGITUD: -0.37966
TEMPERATURA: 15.1 °C
PRECIPITACIÓN: 0.0 mm
ZONA HORARIA: Europe/Madrid


## 2. API de vuestro Flask (obligatorio)
Desde **Jupyter** haz una petición POST JSON a `/api/planificar`. Comprueba: status 200, campos `itinerario`, `meteorologia`, `lugares`, `solicitud_id` y que la ciudad coincide con la enviada. Usa un tipo del conjunto aceptado. Si no tenéis clave Cohere, `LLM_MODE=mock` permite validar transporte y flujo; **no demuestra generación real**.

In [5]:
payload = {
    "ciudad": "Valencia",
    "tipo_turismo": "cultural",
    "duracion_horas": 4,
    "preferencias": "poco caminar",
}
import requests

# URL interna de Docker: comunicación entre los contenedores
api_url = "http://web:5000/api/planificar"

# Realizar petición POST enviando los datos en JSON
respuesta = requests.post(
    api_url,
    json=payload,
    timeout=120
)

# Comprobar el código HTTP
print("STATUS CODE:", respuesta.status_code)

# Comprobar que la petición ha sido correcta
respuesta.raise_for_status()

# Obtener el JSON de respuesta
datos_respuesta = respuesta.json()

# Comprobar los campos obligatorios
campos_obligatorios = [
    "itinerario",
    "meteorologia",
    "lugares",
    "solicitud_id"
]

for campo in campos_obligatorios:
    assert campo in datos_respuesta, f"Falta el campo: {campo}"

# Comprobar que la ciudad coincide con la enviada
assert datos_respuesta["ciudad"].lower() == payload["ciudad"].lower(), \
    "La ciudad devuelta no coincide con la enviada."

print("\nPETICIÓN CORRECTA")
print("Ciudad:", datos_respuesta["ciudad"])
print("Itinerario:", datos_respuesta["itinerario"])
print("Meteorología:", datos_respuesta["meteorologia"])
print("Número de lugares:", len(datos_respuesta["lugares"]))
print("ID de solicitud:", datos_respuesta["solicitud_id"])
print("\nTodas las comprobaciones se han superado.")


STATUS CODE: 200

PETICIÓN CORRECTA
Ciudad: Valencia
Itinerario: [MODO SIMULACIÓN - NO ES IA]
Plan cultural en Valencia:
1. Museo de Ejemplo (DEMO)
2. Jardin de Ejemplo (DEMO)
3. Restaurante de Ejemplo (DEMO)
4. Monumento de Ejemplo (DEMO)
Meteorología: {'codigo_tiempo': 0, 'instante': '2026-11-06T10:15', 'precipitacion_mm': 0.0, 'temperatura_c': 15.1}
Número de lugares: 4
ID de solicitud: 1

Todas las comprobaciones se han superado.


## 3. PostgreSQL (obligatorio)
Después de crear rutas mediante API, conecta a PostgreSQL desde este notebook. Obtén el número de solicitudes y una tabla `id, fecha, ciudad, tipo_turismo, estado` con hasta cinco filas. Compara con el número y últimas filas que ves en vuestro dashboard.

In [7]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(DB, pool_pre_ping=True)

with engine.connect() as conn:

    # 1. Contar solicitudes
    n = conn.execute(
        text("SELECT COUNT(*) FROM solicitudes")
    ).scalar_one()

    print("Filas en PostgreSQL:", n)

    # 2. Consultar las últimas 5 solicitudes
    consulta = text("""
        SELECT id, fecha, ciudad, tipo_turismo, estado
        FROM solicitudes
        ORDER BY fecha DESC
        LIMIT 5
    """)

    filas = conn.execute(consulta).mappings().all()

    # 3. Mostrar resultados
    df_solicitudes = pd.DataFrame(filas)

    print("\nÚltimas solicitudes registradas:")
    display(df_solicitudes)

    print("\nTotal de solicitudes:", n)
    print("Solicitudes mostradas:", len(df_solicitudes))


Filas en PostgreSQL: 1

Últimas solicitudes registradas:


,id,fecha,ciudad,tipo_turismo,estado
0,1,2026-11-06 09:24:46.019884+00:00,Valencia,cultural,exito



Total de solicitudes: 1
Solicitudes mostradas: 1


## 4. Error controlado (obligatorio)
Elige un caso de validación: ciudad vacía, `duracion_horas=99` o `tipo_turismo=espacial`. Haz `POST /api/planificar` y demuestra que la aplicación responde **400** con un mensaje claro. Una respuesta 500 inesperada no supera esta prueba.

In [8]:
import requests

# Enviar una solicitud con una duración no permitida
payload_invalido = {
    "ciudad": "Valencia",
    "tipo_turismo": "cultural",
    "duracion_horas": 99,
    "preferencias": "poco caminar"
}

respuesta_error = requests.post(
    "http://web:5000/api/planificar",
    json=payload_invalido,
    timeout=30
)

# Leer la respuesta
datos_error = respuesta_error.json()

print("STATUS CODE:", respuesta_error.status_code)
print("Mensaje de error:", datos_error.get("error", "Sin mensaje de error"))

# Comprobar que devuelve HTTP 400 y un mensaje claro
assert respuesta_error.status_code == 400, \
    f"Se esperaba HTTP 400, pero se recibió {respuesta_error.status_code}"

assert datos_error.get("error"), \
    "La respuesta no contiene un mensaje de error."

print("\nPRUEBA SUPERADA: la aplicación controla correctamente la entrada inválida.")

STATUS CODE: 400
Mensaje de error: La duración debe ser de 2, 4 o 6 horas.

PRUEBA SUPERADA: la aplicación controla correctamente la entrada inválida.


## 5. Bonus · Analítica desde el notebook
Con `pandas.read_sql_query` y SQLAlchemy, compara las ciudades y tipos más solicitados. Si te interesa, haz aquí una figura de Matplotlib; la figura obligatoria del dashboard **debe seguir saliendo del servidor web y de PostgreSQL**, no copiarse manualmente desde el notebook.

In [ ]:
# BONUS: SELECT ciudad, COUNT(*) ... GROUP BY ciudad ...
# BONUS: pd.read_sql_query(text(...), engine) y una gráfica


### Evidencia final
Guarda el notebook con salidas visibles, sin claves y sin capturas sensibles. En vuestro README, describe brevemente **una conclusión** de los datos almacenados. ¡Cuidado! Las filas `demo=true` son simuladas y se deben identificar como tales.

**Semáforo amarillo:** en tu memoria individual indica si la IA te ayudó a escribir celdas, consultas o asserts; explica qué corregiste y dónde se ve tu prueba. No incluir claves.